# CSMS AI Scoring Engine — OpenAI Juror & Qwen Extractor Pipeline Evaluation

This notebook evaluates the Multi-Model AI Scoring Engine using hybrid model roles:
- **Document Vision Extraction**: Qwen (`Qwen/Qwen3-VL-8B-Instruct` via VLLM REST API)
- **AI Jurors (Compliance, Evidence, HSE)**: OpenAI (`gpt-5-mini` via OpenAI API)
- **Consensus Synthesis Agent**: OpenAI (`gpt-5-mini` via OpenAI API)

It supports:
1. **Single Category Testing (User Selectable)** on `test_submissions/marine` with full evidence resume and juror reasoning output.
2. **Full Batch Testing Across All Categories** on `test_submissions/marine`.

## 1. Setup Environment & Force Reload Engine Modules

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import json
import importlib

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

# Force reload all core engine modules to guarantee latest updates are active
import engine.extractors.pdf_extractor
import engine.extractors.ocr_extractor
import engine.extractors.image_extractor
import engine.extractors.vision_extractor
import engine.extractors.router
import engine.providers.vllm_provider
import engine.providers.openai_provider
import engine.jurors.compliance_juror
import engine.jurors.evidence_juror
import engine.jurors.hse_juror
import engine.consensus.consensus_agent
import engine.pipeline

for mod in [
    engine.extractors.pdf_extractor,
    engine.extractors.ocr_extractor,
    engine.extractors.image_extractor,
    engine.extractors.vision_extractor,
    engine.extractors.router,
    engine.providers.vllm_provider,
    engine.providers.openai_provider,
    engine.jurors.compliance_juror,
    engine.jurors.evidence_juror,
    engine.jurors.hse_juror,
    engine.consensus.consensus_agent,
    engine.pipeline,
]:
    importlib.reload(mod)

t0 = time.time()
from engine.providers.provider_factory import get_provider
print(f"[Step 1 Complete] All engine modules freshly reloaded in {time.time() - t0:.3f}s")

## 2. Verify Providers (Qwen Vision Extractor + OpenAI Jurors)

In [ ]:
t1 = time.time()
roles_file = os.path.join(ROOT_DIR, "configs", "model_roles.json")
with open(roles_file, "r", encoding="utf-8") as f:
    model_roles = json.load(f)

print("Model Roles Configuration:")
print(json.dumps(model_roles, indent=2))

vision_prov = get_provider(role="vision_model")
openai_juror_prov = get_provider(role="openai_reasoning")
openai_cons_prov = get_provider(role="openai_consensus")

print(f"\n[Vision Extractor]     Provider: {vision_prov.__class__.__name__} | Model: {vision_prov.model}")
print(f"[AI Jurors Provider]   Provider: {openai_juror_prov.__class__.__name__} | Model: {openai_juror_prov.model}")
print(f"[Consensus Provider]   Provider: {openai_cons_prov.__class__.__name__} | Model: {openai_cons_prov.model}")

## 3. Single Category Testing Using OpenAI Jurors (`8.1 Sertifikasi Sistem Pengelolaan K3LL` or User Choice)

Change `SELECTED_CATEGORY` below to any folder inside `test_submissions/marine`.

In [ ]:
from engine.pipeline import score_category, discover_category_folders

# Choose category folder name
SELECTED_CATEGORY = "8.1 Sertifikasi Sistem Pengelolaan K3LL"

marine_dir = os.path.join(ROOT_DIR, "test_submissions", "marine")
target_folder = os.path.join(marine_dir, SELECTED_CATEGORY)

if not os.path.exists(target_folder):
    print(f"[WARN] Folder not found: {target_folder}")
    # Fallback to first available category if needed
    available = discover_category_folders(marine_dir)
    if available:
        SELECTED_CATEGORY = available[0]["folder_name"]
        target_folder = available[0]["path"]
        print(f"[INFO] Selected fallback category: {SELECTED_CATEGORY}")

print(f"=== Evaluating Category: {SELECTED_CATEGORY} (Using Qwen Extractor + OpenAI Jurors) ===\n")

t_start = time.time()
single_result = score_category(
    category_id=SELECTED_CATEGORY,
    folder_path=target_folder,
    silent=False,
    juror_role="openai_reasoning",
    consensus_role="openai_consensus"
)
t_elapsed = round(time.time() - t_start, 2)

print(f"\nCompleted single evaluation in {t_elapsed}s")
print(f"Recommended Score : {single_result.get('predicted_score')} / 10")
print(f"Status            : {single_result.get('status').upper()}")

## 4. Full Batch Testing Across All Categories (`test_submissions/marine`) Using OpenAI Jurors

Executes the evaluation pipeline across every discovered marine category folder using Qwen Vision for extraction and OpenAI for Jurors & Consensus.

In [ ]:
from engine.pipeline import evaluate_all_categories

print("=== Starting Full Batch Evaluation Using Qwen Extractor + OpenAI Jurors ===")
batch_t0 = time.time()

batch_report = evaluate_all_categories(
    marine_dir=marine_dir,
    juror_role="openai_reasoning",
    consensus_role="openai_consensus"
)

batch_elapsed = round(time.time() - batch_t0, 2)
print(f"\n[BATCH COMPLETED] Evaluated {batch_report.get('total_categories')} categories in {batch_elapsed}s")
print(f"Passed categories: {batch_report.get('passed')}")
print(f"Failed categories: {batch_report.get('failed')}")
print(f"Overall Accuracy : {batch_report.get('accuracy') * 100:.1f}%")